# Housing Analysis - Refactored Code Demo

This notebook demonstrates the new modular architecture and features of the refactored housing analysis package.

## 1. Setup and Imports

Let's start by importing the new modules and checking the configuration.

In [ ]:
# Standard imports
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# New modular imports
import housing_analysis
from housing_analysis import DataManager, config
from housing_analysis.data import FREDLoader, NYFedLoader
from housing_analysis import utils
from housing_analysis import visualization
from housing_analysis import exceptions

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"Housing Analysis Package Version: {housing_analysis.__version__}")
print(f"Data directory: {config.DATA_DIR}")
print(f"FRED API configured: {config.FRED_API_KEY is not None}")

## 2. Configuration Management

The new `config` module centralizes all configuration settings.

In [ ]:
# Check configuration
print("Configuration Settings:")
print("-" * 50)
print(f"Base Directory: {config.BASE_DIR}")
print(f"Data Directory: {config.DATA_DIR}")
print(f"FRED API URL: {config.FRED_BASE_URL}")
print(f"NY Fed Data URL: {config.NY_FED_MORTGAGE_URL[:50]}...")

# Validate configuration
try:
    config.validate()
    print("\n✅ Configuration is valid!")
except exceptions.ConfigurationError as e:
    print(f"\n❌ Configuration error: {e}")

## 3. Data Loading with DataManager

The `DataManager` provides a unified interface for all data sources.

In [ ]:
# Initialize the data manager
manager = DataManager()

print("Available data loaders:")
print(f"- FRED: {manager.fred.__class__.__name__}")
print(f"- NY Fed: {manager.nyfed.__class__.__name__}")
print(f"- Census: {manager.census.__class__.__name__}")
print(f"\nCache directory: {manager.cache_dir}")

## 4. FRED Data Loading

Load economic data from the Federal Reserve Economic Data (FRED) API.

In [ ]:
# Load unemployment rate data
unemployment = manager.load_unemployment_rate(start_date='2019-01-01')

print(f"Loaded {len(unemployment)} months of unemployment data")
print(f"Date range: {unemployment['date'].min()} to {unemployment['date'].max()}")
print(f"\nFirst 5 rows:")
unemployment.head()

In [ ]:
# Load multiple FRED series
print("Loading multiple economic indicators...")

# Treasury spread (yield curve)
treasury_spread = manager.load_treasury_spread(start_date='2019-01-01')
print(f"✓ Treasury spread: {len(treasury_spread)} observations")

# Mortgage rates
mortgage_rates = manager.load_mortgage_rates(start_date='2019-01-01')
print(f"✓ Mortgage rates: {len(mortgage_rates)} observations")

# Median house price
house_prices = manager.load_median_house_price(start_date='2019-01-01')
print(f"✓ House prices: {len(house_prices)} observations")

print("\nAll data loaded successfully!")

In [ ]:
# Demonstrate the available FRED series
fred_loader = FREDLoader()
available_series = fred_loader.get_available_series()

print("Available FRED data series:")
print("-" * 50)
for series in available_series:
    series_id = fred_loader.SERIES_MAP[series]
    print(f"  {series:25} -> {series_id}")

## 5. Visualization Utilities

The new visualization module provides reusable plotting functions.

In [ ]:
# Load recession data for shading
try:
    recessions = manager.load_recessions()
    print(f"Loaded {len(recessions)} recession periods")
    print("Recent recessions:")
    print(recessions.tail(3)[['Start', 'End']])
except FileNotFoundError:
    print("Recession data file not found. Plots will be shown without recession shading.")
    recessions = None

In [ ]:
# Plot unemployment rate with recession shading
from housing_analysis.visualization import plot_time_series

ax = plot_time_series(
    unemployment,
    column='value',
    title='US Unemployment Rate',
    ylabel='Unemployment Rate (%)',
    recessions=recessions,
    figsize=(14, 6)
)

# Add average line
avg_unemployment = unemployment['value'].mean()
ax.axhline(y=avg_unemployment, color='red', linestyle='--', alpha=0.5, 
           label=f'Average: {avg_unemployment:.1f}%')
ax.legend()
plt.show()

In [ ]:
# Plot treasury spread with fill for negative values
from housing_analysis.visualization import plot_spread_with_fill

ax = plot_spread_with_fill(
    treasury_spread,
    column='value',
    title='10-Year Treasury Minus 3-Month Treasury Spread (Yield Curve)',
    ylabel='Spread (%)',
    recessions=recessions,
    figsize=(14, 6)
)

# Add annotation
ax.text(0.02, 0.95, 'Negative spread indicates yield curve inversion\n(recession predictor)', 
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.show()

## 6. Utility Functions

The utils module provides helpful calculations and data transformations.

In [ ]:
# Calculate year-over-year change for house prices
house_prices['yoy_change'] = utils.calculate_year_over_year_change(house_prices)

# Plot the year-over-year change
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Original prices
ax1.plot(house_prices['date'], house_prices['value'])
ax1.set_title('Median House Prices')
ax1.set_ylabel('Price ($)')
ax1.grid(True, alpha=0.3)

# Year-over-year change
ax2.plot(house_prices['date'], house_prices['yoy_change'])
ax2.axhline(y=0, color='gray', linestyle='--')
ax2.fill_between(house_prices['date'], house_prices['yoy_change'], 0, 
                  where=house_prices['yoy_change']<0, color='red', alpha=0.3)
ax2.set_title('Year-over-Year Change in House Prices')
ax2.set_ylabel('YoY Change (%)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Average YoY change: {house_prices['yoy_change'].mean():.2f}%")
print(f"Max YoY increase: {house_prices['yoy_change'].max():.2f}%")
print(f"Max YoY decrease: {house_prices['yoy_change'].min():.2f}%")

In [ ]:
# Mortgage payment calculator
print("Mortgage Payment Calculator")
print("=" * 50)

# Get latest house price and mortgage rate
latest_price = house_prices['value'].iloc[-1]
latest_rate = mortgage_rates['value'].iloc[-1]

print(f"Current median house price: ${latest_price:,.0f}")
print(f"Current 30-year mortgage rate: {latest_rate:.2f}%")
print()

# Calculate payments for different down payments
down_payments = [0.05, 0.10, 0.20]  # 5%, 10%, 20%

for down_pct in down_payments:
    down_amount = latest_price * down_pct
    loan_amount = latest_price - down_amount
    
    monthly_payment = utils.calculate_mortgage_payment(
        principal=loan_amount,
        rate=latest_rate,
        years=30
    )
    
    print(f"Down payment {down_pct*100:.0f}% (${down_amount:,.0f}):")
    print(f"  Loan amount: ${loan_amount:,.0f}")
    print(f"  Monthly payment: ${monthly_payment:,.2f}")
    print(f"  Annual payment: ${monthly_payment * 12:,.2f}")
    print()

In [ ]:
# Data normalization example
# Normalize house prices and mortgage rates to compare trends

# Align dates for comparison
merged_data = pd.merge(
    house_prices[['date', 'value']].rename(columns={'value': 'house_price'}),
    mortgage_rates[['date', 'value']].rename(columns={'value': 'mortgage_rate'}),
    on='date',
    how='inner'
)

# Normalize both series to 100 at the start
merged_data['house_price_norm'] = utils.normalize_series(
    merged_data[['date', 'house_price']].rename(columns={'house_price': 'value'}),
    value_column='value'
)

merged_data['mortgage_rate_norm'] = utils.normalize_series(
    merged_data[['date', 'mortgage_rate']].rename(columns={'mortgage_rate': 'value'}),
    value_column='value'
)

# Plot normalized comparison
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(merged_data['date'], merged_data['house_price_norm'], 
        label='House Prices', linewidth=2)
ax.plot(merged_data['date'], merged_data['mortgage_rate_norm'], 
        label='Mortgage Rates', linewidth=2)

ax.set_title('Normalized Comparison: House Prices vs Mortgage Rates (Index: Start = 100)')
ax.set_ylabel('Index Value')
ax.axhline(y=100, color='gray', linestyle='--', alpha=0.5)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"House price change: {merged_data['house_price_norm'].iloc[-1] - 100:.1f}%")
print(f"Mortgage rate change: {merged_data['mortgage_rate_norm'].iloc[-1] - 100:.1f}%")

## 7. NY Fed Data Loading

Load household debt and credit data from the New York Fed.

In [ ]:
# Check if NY Fed data is available
ny_fed_file = config.DATA_DIR / 'HHD_C_Report_2024Q4.xlsx'

if ny_fed_file.exists():
    print("NY Fed data file found!")
    
    # Load loan report data
    loan_data, title, value_label = manager.load_loan_report()
    
    print(f"\nTitle: {title}")
    print(f"Value Label: {value_label}")
    print(f"\nData shape: {loan_data.shape}")
    print(f"Columns: {', '.join(loan_data.columns[:5])}...")
    print(f"\nFirst few rows:")
    display(loan_data.head())
    
    # Plot mortgage debt over time
    if 'Mortgage' in loan_data.columns:
        fig, ax = plt.subplots(figsize=(14, 6))
        ax.plot(loan_data['date'], loan_data['Mortgage'], linewidth=2)
        ax.set_title(f'{title} - Mortgage Debt')
        ax.set_ylabel(value_label)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print(f"NY Fed data file not found at: {ny_fed_file}")
    print("The NY Fed loader will download it automatically when first used.")
    print("\nAvailable NY Fed data types:")
    nyfed_loader = NYFedLoader()
    for series in nyfed_loader.get_available_series():
        print(f"  - {series}")

## 8. Error Handling

The new exception hierarchy provides better error handling.

In [ ]:
# Demonstrate error handling
from housing_analysis.exceptions import (
    DataLoadError, APIError, ConfigurationError, DataNotFoundError
)

# Example 1: Handle missing data
try:
    fred_loader = FREDLoader()
    # Try to load non-existent series
    data = fred_loader.load('NONEXISTENT_SERIES')
except DataLoadError as e:
    print(f"Data load error handled: {e}")
except Exception as e:
    print(f"Unexpected error: {e}")

print()

# Example 2: Configuration validation
try:
    # Temporarily remove API key
    original_key = config.FRED_API_KEY
    config.FRED_API_KEY = None
    config.validate()
except ValueError as e:
    print(f"Configuration error handled: {e}")
finally:
    # Restore API key
    config.FRED_API_KEY = original_key

print("\nError handling works correctly!")

## 9. Caching Performance

The new system includes automatic caching for better performance.

In [ ]:
import time

# Test caching performance
fred_loader = FREDLoader(cache_dir=config.DATA_DIR / 'cache')

# First load (from API)
start_time = time.time()
data1 = fred_loader.load('unemployment_rate', start_date='2020-01-01', use_cache=False)
api_time = time.time() - start_time
print(f"Loading from API: {api_time:.3f} seconds")

# Second load (from cache)
start_time = time.time()
data2 = fred_loader.load('unemployment_rate', start_date='2020-01-01', use_cache=True)
cache_time = time.time() - start_time
print(f"Loading from cache: {cache_time:.3f} seconds")

if api_time > 0:
    speedup = api_time / cache_time
    print(f"\nCache speedup: {speedup:.1f}x faster")

# Verify data is identical
print(f"\nData identical: {data1.equals(data2)}")

## 10. Backward Compatibility

The old `Datasets` class still works for compatibility with existing code.

In [ ]:
# Old way still works
from housing_analysis import Datasets

old_datasets = Datasets()

print("Old Datasets class methods:")
methods = [m for m in dir(old_datasets) if not m.startswith('_') and callable(getattr(old_datasets, m))]
for i in range(0, len(methods), 3):
    print(f"  {', '.join(methods[i:i+3])}")

print("\n✅ Backward compatibility maintained!")

## Summary

The refactored housing analysis package provides:

### ✨ Key Benefits
- **Modular Architecture**: Clean separation of concerns
- **Centralized Configuration**: Single source of truth for settings
- **Reusable Components**: Utilities and visualizations can be used anywhere
- **Better Error Handling**: Custom exceptions for clearer debugging
- **Performance**: Automatic caching for faster data access
- **Type Safety**: Full type hints for better IDE support
- **Testing**: Comprehensive test suite ensures reliability
- **Backward Compatibility**: Existing code continues to work

### 📚 Next Steps
- Explore the individual modules for more features
- Check the test files for usage examples
- Read the REFACTORING_COMPLETE.md for full documentation
- Gradually migrate existing notebooks to use the new interfaces